Pop-wtd, global temperature anomaly for Aug-Jan. IPCC recent baseline period (1995-2014)

In [1]:
import os
from dotenv import load_dotenv
import xarray as xr

import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

import analysis_utils
import isku_utils

import importlib

importlib.reload(analysis_utils)
importlib.reload(isku_utils)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'isku_utils' from '/home/emily_zuetell/projects/poreallas/analysis/isku_utils.py'>

In [2]:
load_dotenv()
DATA_DIR = os.environ["DATA_DIR"]
IMPACT_REGION_POLYGONS = os.environ["POREALLAS_REGIONS_POLYGONS_URI"]
SOCIOECONOMICS_URI = os.environ["POREALLAS_SOCIOECONOMICS_URI"]


In [ ]:
# Impact Regions
_polygons = (
    gpd.read_parquet(os.path.join(DATA_DIR, IMPACT_REGION_POLYGONS))
    .rename(columns={"hierid": "region"})
    .set_index("region")
    .set_crs(epsg=4326)  # Assuming the data is WGS-82.
)

# Socioeconomics
socioeconomics = xr.open_zarr(os.path.join(DATA_DIR, SOCIOECONOMICS_URI))
socioeconomics = socioeconomics.sel(year=2026)[
    ["pop", "gdppc", "iso3"]
]

In [ ]:
# ECMWF Forecast and ERA5 baseline adjusted to GMFD
POREALLAS_ERA5_URI = "gs://poreallas-public-20260605/v20260825/parsed/era5_adj-q100.zarr"
POREALLAS_TAS_FORECAST_URI = "gs://poreallas-public-20260605/v20260825/parsed/forecast_adj-q100.zarr"

forecast = xr.load_dataset(
        POREALLAS_TAS_FORECAST_URI,
        engine="zarr",
        chunks={},
        backend_kwargs={"storage_options": {"token": "anon"}},
    )

In [9]:
era5= xr.load_dataset(
        POREALLAS_ERA5_URI,
        engine="zarr",
        chunks={},
        backend_kwargs={"storage_options": {"token": "anon"}},
    )

In [ ]:
# Monthly Mean IPCC Recent Baseline
era5_baseline = (era5.sel(time=slice("1995-01-01", "2014-12-31"))
                 .groupby("time.month")
                 .mean()
                 .sel(month=[1, 8, 9, 10, 11, 12]))
# Monthly Forecast Anomaly
anomaly = (forecast.sel(time=slice("2026-08-01", "2027-01-31")).groupby("time.month").mean())-era5_baseline

In [ ]:
# Convert Ensemble Average to Impact Regions
anomaly_ir = isku_utils.grid_to_ir(anomaly.mean(dim = 'month'), savefile=None)
_polygons_anomaly = analysis_utils.xarray_to_gpd(
    anomaly_ir['tas'].mean(dim="number"), _polygons
)

/home/emily_zuetell/projects/poreallas/analysis/isku_utils.py:29: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  _region_weights = xr.load_dataset(uri)[
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/core/group.py:3289: ZarrUserWarning: Object at zarr.json:Zone.Identifier is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


In [25]:
merged = _polygons_anomaly.merge(socioeconomics.to_dataframe(), left_on="region", right_on="region")

In [ ]:
pop_weighted_anomaly = (merged['tas']*merged['pop']).sum()/merged['pop'].sum()

np.float64(1.0222095645728726)